# 初期モデル構築 - テーブルデータのみ

衛星データなしでテーブルデータのみを使用して、LightGBM と CatBoost で建設コスト（USD/m²）を予測します。
評価指標: RMSLE / K-Fold (k=3) クロスバリデーション

## 実験設定\n\n**`CASE_NAME` を変更すると対応する `conf/parameter_{CASE_NAME}.yml` が読み込まれる。**"

In [1]:
CASE_NAME = 'initial_tabular_only'

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns=200
sys.path.insert(0, '..')
from src.config import load_config
from src.features import TARGET, CAT_COLS, prepare_features
from src.training import rmsle, run_cv, make_lgb_fn, make_cb_fn
from src.io import save_submission

cfg = load_config(CASE_NAME)

TRAIN_CSV       = cfg['train_csv']
EVAL_CSV        = cfg['eval_csv']
SEED            = cfg['seed']
N_SPLITS        = cfg['n_splits']
EXTRA_DROP_COLS = cfg.get('extra_drop_cols')
LGB_PARAMS      = cfg['lgb_params']
CB_PARAMS       = cfg['cb_params']

RESULT_DIR = f'../data/result/{CASE_NAME}'
os.makedirs(RESULT_DIR, exist_ok=True)
print(f'Loaded: conf/parameter_{CASE_NAME}.yml')
print(f'Result directory: {RESULT_DIR}')

Loaded: conf/parameter_initial_tabular_only.yml
Result directory: ../data/result/initial_tabular_only


## データ読み込み

In [3]:
train_df = pd.read_csv(TRAIN_CSV)
eval_df  = pd.read_csv(EVAL_CSV)

print(f'Train shape: {train_df.shape}')
print(f'Eval  shape: {eval_df.shape}')
train_df.head(3)

Train shape: (1024, 23)
Eval  shape: (1024, 22)


,data_id,geolocation_name,quarter_label,country,year,deflated_gdp_usd,us_cpi,developed_country,landlocked,region_economic_classification,access_to_airport,access_to_port,access_to_highway,access_to_railway,straight_distance_to_capital_km,seismic_hazard_zone,flood_risk_class,tropical_cyclone_wind_risk,tornadoes_wind_risk,koppen_climate_zone,sentinel2_tiff_file_name,viirs_tiff_file_name,construction_cost_per_m2_usd
0,LP81L,Dinagat Islands,2019-Q3,Philippines,2019,2.996821e+11,117.689915,No,No,Lower-middle income,No,Yes,Yes,No,770.0,Moderate,Yes,High,Very Low,Af,sentinel_2_dinagat_islands_2019-Q3.tif,viirs_dinagat_islands_2019-Q3.tif,129.997420
1,E7EOB,29000 Nara,2024-Q2,Japan,2024,3.928801e+12,143.968241,Yes,Yes,High income,No,No,Yes,Yes,370.0,Moderate,Yes,Low,Very Low,Cfa,sentinel_2_29000_nara_2024-Q2.tif,viirs_29000_nara_2024-Q2.tif,1567.878774
2,WAOUA,05000 Akita,2020-Q1,Japan,2020,4.069008e+12,118.435291,Yes,No,High income,Yes,Yes,Yes,Yes,450.0,Moderate,Yes,Low,Very Low,Dfa,sentinel_2_05000_akita_2020-Q1.tif,viirs_05000_akita_2020-Q1.tif,2009.827701


## 特徴量エンジニアリング

In [4]:
X_train = prepare_features(train_df, extra_drop_cols=EXTRA_DROP_COLS)
y_train = train_df[TARGET].copy()

# eval CSV に train と同じ列がない場合は None にして CV のみ実行
try:
    X_eval = prepare_features(eval_df, extra_drop_cols=EXTRA_DROP_COLS)
    if not set(X_train.columns) <= set(X_eval.columns):
        raise ValueError('eval に存在しない列があります。')
except (KeyError, ValueError) as e:
    print(f'WARNING: X_eval を None にします ({e})')
    X_eval = None

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_eval : {X_eval.shape if X_eval is not None else None}')
print('Features:', X_train.columns.tolist())

X_train: (1024, 19), y_train: (1024,)
X_eval : (1024, 19)
Features: ['geolocation_name', 'country', 'year', 'deflated_gdp_usd', 'us_cpi', 'developed_country', 'landlocked', 'region_economic_classification', 'access_to_airport', 'access_to_port', 'access_to_highway', 'access_to_railway', 'straight_distance_to_capital_km', 'seismic_hazard_zone', 'flood_risk_class', 'tropical_cyclone_wind_risk', 'tornadoes_wind_risk', 'koppen_climate_zone', 'quarter_num']


In [5]:
X_train

,geolocation_name,country,year,deflated_gdp_usd,us_cpi,developed_country,landlocked,region_economic_classification,access_to_airport,access_to_port,access_to_highway,access_to_railway,straight_distance_to_capital_km,seismic_hazard_zone,flood_risk_class,tropical_cyclone_wind_risk,tornadoes_wind_risk,koppen_climate_zone,quarter_num
0,Dinagat Islands,Philippines,2019,2.996821e+11,117.689915,No,No,Lower-middle income,No,Yes,Yes,No,770.0,Moderate,Yes,High,Very Low,Af,3
1,29000 Nara,Japan,2024,3.928801e+12,143.968241,Yes,Yes,High income,No,No,Yes,Yes,370.0,Moderate,Yes,Low,Very Low,Cfa,2
2,05000 Akita,Japan,2020,4.069008e+12,118.435291,Yes,No,High income,Yes,Yes,Yes,Yes,450.0,Moderate,Yes,Low,Very Low,Dfa,1
3,Cotabato,Philippines,2020,2.912443e+11,119.402476,No,Yes,Lower-middle income,Yes,No,Yes,No,870.0,High,Yes,Moderate,Very Low,Af,4
4,Pampanga,Philippines,2019,2.996821e+11,117.689915,No,Yes,Lower-middle income,Yes,No,Yes,Yes,65.0,Moderate,Yes,Moderate,Very Low,Am,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1019,Quirino,Philippines,2021,3.321833e+11,125.484873,No,Yes,Lower-middle income,No,No,Yes,No,265.0,Moderate,Yes,Moderate,Very Low,Am,3
1020,Bulacan,Philippines,2021,3.321833e+11,127.389434,No,No,Lower-middle income,No,No,Yes,Yes,30.0,Moderate,Yes,High,Very Low,Am,4
1021,24000 Mie,Japan,2020,4.069008e+12,119.128540,Yes,No,High income,No,Yes,Yes,Yes,320.0,High,Yes,Moderate,Low,Cfa,3
1022,06000 Yamagata,Japan,2023,3.993447e+12,140.861386,Yes,Yes,High income,Yes,No,Yes,Yes,300.0,Moderate,Yes,Low,Very Low,Dfa,4


## クロスバリデーション実行

In [6]:
print('[LightGBM]')
lgb_oof, lgb_eval, lgb_score = run_cv(
    make_lgb_fn(LGB_PARAMS, CAT_COLS), X_train, y_train, X_eval,
    n_splits=N_SPLITS, seed=SEED,
)

print('\n[CatBoost]')
cb_oof, cb_eval, cb_score = run_cv(
    make_cb_fn(CB_PARAMS, CAT_COLS), X_train, y_train, X_eval,
    n_splits=N_SPLITS, seed=SEED,
)

[LightGBM]
[200]	valid_0's rmse: 185.839
  Fold 1 RMSLE: 0.1990
[200]	valid_0's rmse: 151.653
  Fold 2 RMSLE: 0.2471
[200]	valid_0's rmse: 164.147
  Fold 3 RMSLE: 0.2150
  OOF RMSLE: 0.2212 | Mean Fold: 0.2204

[CatBoost]
0:	learn: 798.6015682	test: 801.0681619	best: 801.0681619 (0)	total: 67.6ms	remaining: 1m 7s
200:	learn: 131.1835214	test: 188.6107474	best: 188.6107474 (200)	total: 513ms	remaining: 2.04s
400:	learn: 102.1708538	test: 179.7028479	best: 179.7028479 (400)	total: 894ms	remaining: 1.33s
600:	learn: 84.4767613	test: 177.2242933	best: 177.1200392 (587)	total: 1.29s	remaining: 857ms
Stopped by overfitting detector  (50 iterations wait)

bestTest = 176.4911184
bestIteration = 708

Shrink model to first 709 iterations.
  Fold 1 RMSLE: 0.1970
0:	learn: 808.9168615	test: 784.6815950	best: 784.6815950 (0)	total: 3.45ms	remaining: 3.44s
200:	learn: 158.6740126	test: 149.1178047	best: 149.1178047 (200)	total: 395ms	remaining: 1.57s
400:	learn: 126.4572717	test: 144.3294351	best: 1

## 結果サマリ

In [7]:
ensemble_oof   = (lgb_oof + cb_oof) / 2
ensemble_score = rmsle(y_train, ensemble_oof)

print('=' * 40)
print('モデル比較 (3-Fold CV)')
print('=' * 40)
print(f'LightGBM OOF RMSLE : {lgb_score:.4f}')
print(f'CatBoost OOF RMSLE : {cb_score:.4f}')
print(f'Ensemble OOF RMSLE : {ensemble_score:.4f}')

モデル比較 (3-Fold CV)
LightGBM OOF RMSLE : 0.2212
CatBoost OOF RMSLE : 0.2108
Ensemble OOF RMSLE : 0.2070


## 予測結果の保存

In [8]:
from src.io import save_oof

# OOF 予測を元テーブルにマージして保存
oof_df = save_oof(
    oof_preds={'lgb': lgb_oof, 'cb': cb_oof, 'ensemble': ensemble_oof},
    train_df=train_df,
    result_dir=RESULT_DIR,
)

# スコアの保存
pd.DataFrame({
    'model':     ['LightGBM', 'CatBoost', 'Ensemble'],
    'oof_rmsle': [lgb_score,  cb_score,   ensemble_score],
}).to_csv(f'{RESULT_DIR}/scores.csv', index=False)
print(f'Saved -> {RESULT_DIR}/scores.csv')

# 提出ファイルの保存（X_eval が揃っている場合のみ）
if lgb_eval is not None and cb_eval is not None:
    ensemble_eval = (lgb_eval + cb_eval) / 2
    save_submission(lgb_eval,     'submission_lgb.csv',      eval_df, RESULT_DIR)
    save_submission(cb_eval,      'submission_cb.csv',       eval_df, RESULT_DIR)
    ens_sub = save_submission(ensemble_eval, 'submission_ensemble.csv', eval_df, RESULT_DIR)
    display(ens_sub.head())
else:
    print('X_eval=None のため提出ファイルの保存をスキップしました。')

oof_df.head()

Saved -> ../data/result/initial_tabular_only/oof_predictions.csv
Saved -> ../data/result/initial_tabular_only/scores.csv
Saved -> ../data/result/initial_tabular_only/submission_lgb.csv
Saved -> ../data/result/initial_tabular_only/submission_cb.csv
Saved -> ../data/result/initial_tabular_only/submission_ensemble.csv


,data_id,construction_cost_per_m2_usd
0,3TOW4,1834.884019
1,493WX,1626.371802
2,UYP04,1749.473235
3,FN33V,242.101912
4,CPRHV,1694.355433


,data_id,geolocation_name,quarter_label,country,year,deflated_gdp_usd,us_cpi,developed_country,landlocked,region_economic_classification,access_to_airport,access_to_port,access_to_highway,access_to_railway,straight_distance_to_capital_km,seismic_hazard_zone,flood_risk_class,tropical_cyclone_wind_risk,tornadoes_wind_risk,koppen_climate_zone,sentinel2_tiff_file_name,viirs_tiff_file_name,construction_cost_per_m2_usd,oof_lgb,oof_cb,oof_ensemble
0,LP81L,Dinagat Islands,2019-Q3,Philippines,2019,2.996821e+11,117.689915,No,No,Lower-middle income,No,Yes,Yes,No,770.0,Moderate,Yes,High,Very Low,Af,sentinel_2_dinagat_islands_2019-Q3.tif,viirs_dinagat_islands_2019-Q3.tif,129.997420,149.161891,176.941535,163.051713
1,E7EOB,29000 Nara,2024-Q2,Japan,2024,3.928801e+12,143.968241,Yes,Yes,High income,No,No,Yes,Yes,370.0,Moderate,Yes,Low,Very Low,Cfa,sentinel_2_29000_nara_2024-Q2.tif,viirs_29000_nara_2024-Q2.tif,1567.878774,1665.571008,1702.031948,1683.801478
2,WAOUA,05000 Akita,2020-Q1,Japan,2020,4.069008e+12,118.435291,Yes,No,High income,Yes,Yes,Yes,Yes,450.0,Moderate,Yes,Low,Very Low,Dfa,sentinel_2_05000_akita_2020-Q1.tif,viirs_05000_akita_2020-Q1.tif,2009.827701,1771.960798,1764.251509,1768.106153
3,2IZ5P,Cotabato,2020-Q4,Philippines,2020,2.912443e+11,119.402476,No,Yes,Lower-middle income,Yes,No,Yes,No,870.0,High,Yes,Moderate,Very Low,Af,sentinel_2_cotabato_2020-Q4.tif,viirs_cotabato_2020-Q4.tif,377.279961,249.982600,249.210924,249.596762
4,RJ5XF,Pampanga,2019-Q3,Philippines,2019,2.996821e+11,117.689915,No,Yes,Lower-middle income,Yes,No,Yes,Yes,65.0,Moderate,Yes,Moderate,Very Low,Am,sentinel_2_pampanga_2019-Q3.tif,viirs_pampanga_2019-Q3.tif,163.905688,209.746259,190.917742,200.332000
